# Step 6: Policy interpretation -- what did the deep hedging model actually learn?

We already have strong quantitative evidence (n_trades stayed at ~20/24, same as BS delta) that the model did NOT learn no-trade-band-like behavior. This notebook makes that visually concrete: pick one test episode, plot BS delta, Whalley-Wilmott's band+position, and the deep hedge model's position, all together.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
from scipy.stats import norm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BTC_TRANSACTION_COST_RATE = 0.0005
RISK_AVERSION_LAMBDA = 60
norm_stats = json.load(open("norm_stats.json"))
feature_names = ["moneyness", "ttm", "iv", "delta", "is_call"]

def bs_delta_fn(S, K, T, sigma, option_type, r=0.0):
    if T <= 0 or sigma <= 0:
        return (1.0 if S > K else 0.0) if option_type == "call" else (-1.0 if S < K else 0.0)
    d1 = (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1) if option_type == "call" else norm.cdf(d1) - 1.0

def bs_gamma_fn(S, K, T, sigma):
    if T <= 0 or sigma <= 0:
        return 0.0
    d1 = (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    return norm.pdf(d1) / (S * sigma * np.sqrt(T))

def ww_band(S, gamma, k, lam, T):
    if gamma <= 0:
        return 0.0
    return ((3 * k * S**2 * gamma**2) / (2 * lam)) ** (1/3)

test = pd.read_csv("btc_options_test.csv")
test["hour_bucket"] = pd.to_datetime(test["hour_bucket"])
test["sample_date"] = test["hour_bucket"].dt.date
test["T_years"] = test["time_to_maturity_days"] / 365
test["iv_decimal"] = test["mark_iv"] / 100
test["option_mid_usd"] = test["mid_price"] * test["underlying_price"]

# Pick an episode with plenty of steps and near-the-money (where WW's band and any model behavior is most visible)
test["moneyness"] = test["underlying_price"] / test["strike_price"]
candidates = test[(test["moneyness"] > 0.9) & (test["moneyness"] < 1.1)]
counts = candidates.groupby(["symbol", "sample_date"]).size().sort_values(ascending=False)
viz_symbol, viz_date = counts.index[0]
print(f"Visualizing: {viz_symbol} on {viz_date}, {counts.iloc[0]} steps")

ep = test[(test["symbol"] == viz_symbol) & (test["sample_date"] == viz_date)].sort_values("hour_bucket").reset_index(drop=True)

## Compute BS delta, WW band, and deep hedge positions for this one episode

In [ ]:
n = len(ep)
bs_deltas = np.array([bs_delta_fn(ep["underlying_price"].iloc[i], ep["strike_price"].iloc[i],
                                    ep["T_years"].iloc[i], ep["iv_decimal"].iloc[i], ep["type"].iloc[i]) for i in range(n)])
gammas = np.array([bs_gamma_fn(ep["underlying_price"].iloc[i], ep["strike_price"].iloc[i],
                                 ep["T_years"].iloc[i], ep["iv_decimal"].iloc[i]) for i in range(n)])
bands = np.array([ww_band(ep["underlying_price"].iloc[i], gammas[i], BTC_TRANSACTION_COST_RATE,
                            RISK_AVERSION_LAMBDA, ep["T_years"].iloc[i]) for i in range(n)])

# WW simulated position
ww_position = np.zeros(n)
current = 0.0
for i in range(n):
    lower, upper = bs_deltas[i] - bands[i], bs_deltas[i] + bands[i]
    if i == 0:
        new_pos = bs_deltas[i]
    elif current < lower:
        new_pos = lower
    elif current > upper:
        new_pos = upper
    else:
        new_pos = current
    current = new_pos
    ww_position[i] = current

# Deep hedge v2 position
class DeepHedgePolicy(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        raw_position = self.head(lstm_out)
        return 1.5 * torch.tanh(raw_position.squeeze(-1))

model = DeepHedgePolicy().to(device)
model.load_state_dict(torch.load("best_deep_hedge_model_v2.pt", map_location=device))
model.eval()

moneyness_vals = (ep["underlying_price"] / ep["strike_price"]).values
ttm_vals = ep["T_years"].values
iv_vals = ep["iv_decimal"].clip(upper=3.0).values
is_call_vals = (ep["type"] == "call").astype(float).values
feat = np.stack([moneyness_vals, ttm_vals, iv_vals, bs_deltas, is_call_vals], axis=1)
for i, name in enumerate(feature_names):
    feat[:, i] = (feat[:, i] - norm_stats[name]["mean"]) / norm_stats[name]["std"]
feat_padded = np.zeros((24, 5))
feat_padded[:n] = feat
feat_tensor = torch.tensor(feat_padded, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    deep_hedge_position = model(feat_tensor).cpu().numpy()[0, :n]

print("Computed BS delta, WW band+position, and deep hedge v2 position for this episode.")

## The key comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
hours = ep["hour_bucket"]

ax.plot(hours, bs_deltas, label="BS delta (target)", color="gray", linestyle="--")
ax.fill_between(hours, bs_deltas - bands, bs_deltas + bands, alpha=0.15, color="blue", label="WW no-trade band")
ax.plot(hours, ww_position, label="Whalley-Wilmott position", color="darkblue", marker="s", markersize=4)
ax.plot(hours, deep_hedge_position, label="Deep hedge (v2) position", color="darkred", marker="o", markersize=4)

ax.legend()
ax.set_title(f"Policy comparison: {viz_symbol} on {viz_date}\nWW makes {(np.abs(np.diff(ww_position, prepend=0))>1e-6).sum()} trades, Deep hedge makes {(np.abs(np.diff(deep_hedge_position, prepend=0))>1e-6).sum()} trades (out of {n} steps)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()